Table of Contents <a id="content"></a>
1. [Introduction to Computer Vision](#intro)

1. [Image Fundamentals](#image_fundamentals)

1. [Image Processing Basics](#image-processing-basics)

1. [Image Filtering](#image_filtering)

1. [Edge Detection](#edge_detection)

1. [Feature Detection](#feature_detection)

1. [Feature Matching](#feature_matching)

1. [Geometric Transformations](#geometric_transformations)

1. [Image Segmentation](#image_segmentation)

1. [Object Detection](#object_detection)

1. [Motion Analysis](#motion_analysis)

# Introduction to Computer Vision <a id="intro"></a>
Go to [content](#content)

### What is Traditional Computer Vision?
Computer vision is the field of study that enables computers to interpret and understand visual information from the world. Traditional computer vision refers to techniques that rely on mathematical models, statistical methods, and image processing algorithms rather than deep learning.



### Prerequisites
Basic Python programming

Understanding of linear algebra

Basic calculus knowledge

Familiarity with NumPy

### Setup

In [ ]:
# Install required libraries
# pip install numpy opencv-python matplotlib scikit-image scipy

import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import io, color, filters, feature
import warnings
warnings.filterwarnings('ignore')

# Image Fundamentals <a id="image_fundamentals"></a>
Go to [content](#content)

1. Image Representation

In [1]:
# Create sample images
# Grayscale image (2D array)
gray_img = np.random.randint(0, 255, size=(100, 100), dtype=np.uint8)

# Color image (3D array - height, width, channels)
color_img = np.random.randint(0, 255, size=(100, 100, 3), dtype=np.uint8)

# Binary image (0 and 1)
binary_img = np.random.randint(0, 2, size=(100, 100), dtype=np.uint8)

print(f"Grayscale shape: {gray_img.shape}")
print(f"Color shape: {color_img.shape}")
print(f"Binary shape: {binary_img.shape}")

NameError: name 'np' is not defined

2. Loading and Displaying Images

In [ ]:
# Load image using OpenCV
img_path = 'sample.jpg'  # Replace with your image path
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB

# Display images
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_rgb)
axes[0].set_title('Original Image')
axes[0].axis('off')

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
axes[1].imshow(gray, cmap='gray')
axes[1].set_title('Grayscale Image')
axes[1].axis('off')

# Display color channels
b, g, r = cv2.split(img_rgb)
axes[2].imshow(np.stack([r, np.zeros_like(r), np.zeros_like(r)], axis=2))
axes[2].set_title('Red Channel')
axes[2].axis('off')
plt.show()

### 3. Pixel Operations

In [ ]:
# Basic pixel operations
def pixel_operations(image):
    # Access and modify pixel values
    height, width = image.shape[:2]
    
    # Get pixel value
    pixel = image[100, 100]
    print(f"Pixel at (100,100): {pixel}")
    
    # Set pixel value
    image[100, 100] = [255, 255, 255]
    
    # Region of Interest (ROI)
    roi = image[50:150, 50:150]
    return roi

# Brightness adjustment
def adjust_brightness(image, factor):
    adjusted = np.clip(image * factor, 0, 255).astype(np.uint8)
    return adjusted

# Contrast adjustment
def adjust_contrast(image, alpha):
    adjusted = np.clip(alpha * image, 0, 255).astype(np.uint8)
    return adjusted

# Image Processing Basics <a id="image-processing-basics"></a>
Go to [content](#content)

### 1. Histogram Operations

In [ ]:
def histogram_demo(image):
    # Compute histogram
    if len(image.shape) == 3:
        # Color image
        colors = ('b', 'g', 'r')
        for i, color in enumerate(colors):
            hist = cv2.calcHist([image], [i], None, [256], [0, 256])
            plt.plot(hist, color=color)
    else:
        # Grayscale image
        hist = cv2.calcHist([image], [0], None, [256], [0, 256])
        plt.plot(hist, color='black')
    
    plt.title('Image Histogram')
    plt.xlabel('Pixel Intensity')
    plt.ylabel('Frequency')
    plt.show()
    
    return hist

# Histogram equalization
def histogram_equalization(image):
    if len(image.shape) == 3:
        # Apply to each channel
        ycrcb = cv2.cvtColor(image, cv2.COLOR_RGB2YCrCb)
        ycrcb[:,:,0] = cv2.equalizeHist(ycrcb[:,:,0])
        equalized = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)
    else:
        equalized = cv2.equalizeHist(image)
    return equalized

### 2. Thresholding

In [ ]:
def thresholding_demo(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    
    # Simple threshold
    ret, thresh_simple = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    
    # Adaptive threshold
    thresh_adaptive = cv2.adaptiveThreshold(gray, 255, 
                                           cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                           cv2.THRESH_BINARY, 11, 2)
    
    # Otsu's threshold
    ret, thresh_otsu = cv2.threshold(gray, 0, 255, 
                                     cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    images = [gray, thresh_simple, thresh_adaptive, thresh_otsu]
    titles = ['Original', 'Simple Threshold', 'Adaptive Threshold', 'Otsu Threshold']
    
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    plt.show()

# Image Filtering <a id="image_filtering"></a>
Go to [content](#content)

### 1. Spatial Domain Filtering

In [ ]:
def filtering_demo(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    
    # Add some noise
    noise = np.random.normal(0, 25, gray.shape).astype(np.uint8)
    noisy = cv2.add(gray, noise)
    
    # Mean filter
    mean_filtered = cv2.blur(noisy, (5, 5))
    
    # Gaussian filter
    gaussian_filtered = cv2.GaussianBlur(noisy, (5, 5), 0)
    
    # Median filter (good for salt & pepper noise)
    median_filtered = cv2.medianBlur(noisy, 5)
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    images = [noisy, mean_filtered, gaussian_filtered, median_filtered]
    titles = ['Noisy Image', 'Mean Filter', 'Gaussian Filter', 'Median Filter']
    
    for ax, img, title in zip(axes.flat, images, titles):
        ax.imshow(img, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    plt.show()

### 2. Frequency Domain Filtering

In [ ]:
def frequency_filtering(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    
    # Fourier Transform
    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    
    # Create ideal low-pass filter
    rows, cols = gray.shape
    crow, ccol = rows//2, cols//2
    
    # Create a mask
    mask = np.zeros((rows, cols), np.uint8)
    r = 30  # radius
    mask[crow-r:crow+r, ccol-r:ccol+r] = 1
    
    # Apply mask
    fshift_filtered = fshift * mask
    
    # Inverse transform
    f_ishift = np.fft.ifftshift(fshift_filtered)
    img_back = np.fft.ifft2(f_ishift)
    img_back = np.real(img_back)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(gray, cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    # Show magnitude spectrum
    magnitude_spectrum = 20*np.log(np.abs(fshift) + 1)
    axes[1].imshow(magnitude_spectrum, cmap='gray')
    axes[1].set_title('Magnitude Spectrum')
    axes[1].axis('off')
    
    axes[2].imshow(img_back, cmap='gray')
    axes[2].set_title('Low-pass Filtered')
    axes[2].axis('off')
    plt.show()

# Edge detection <a id="edge_detection"></a>
Go to [content](#content)

### 1. Gradient-based Edge Detection

In [ ]:
def edge_detection_demo(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    
    # Sobel operator
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel = np.sqrt(sobelx**2 + sobely**2)
    sobel = np.clip(sobel, 0, 255).astype(np.uint8)
    
    # Laplacian
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    laplacian = np.uint8(np.absolute(laplacian))
    
    # Canny edge detection
    edges = cv2.Canny(gray, 50, 150)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    images = [gray, sobel, laplacian, edges]
    titles = ['Original', 'Sobel', 'Laplacian', 'Canny']
    
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    plt.show()
    
    return edges

### 2. Advanced Edge Detection

In [ ]:
def advanced_edges(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    
    # Canny with different parameters
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Low threshold
    edges_low = cv2.Canny(gray, 30, 100)
    axes[0].imshow(edges_low, cmap='gray')
    axes[0].set_title('Low Threshold (30,100)')
    axes[0].axis('off')
    
    # Medium threshold
    edges_med = cv2.Canny(gray, 50, 150)
    axes[1].imshow(edges_med, cmap='gray')
    axes[1].set_title('Medium Threshold (50,150)')
    axes[1].axis('off')
    
    # High threshold
    edges_high = cv2.Canny(gray, 100, 200)
    axes[2].imshow(edges_high, cmap='gray')
    axes[2].set_title('High Threshold (100,200)')
    axes[2].axis('off')
    plt.show()

# Feature Detection <a id="feature_detection"></a>
Go to [content](#content)

### 1. Corner Detection

In [ ]:
def corner_detection(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    
    # Harris corner detection
    harris = cv2.cornerHarris(gray, 2, 3, 0.04)
    harris = cv2.dilate(harris, None)  # Dilate for visualization
    
    # Threshold for corners
    img_harris = image.copy()
    img_harris[harris > 0.01*harris.max()] = [255, 0, 0]  # Mark corners in red
    
    # Shi-Tomasi corners
    corners = cv2.goodFeaturesToTrack(gray, 25, 0.01, 10)
    corners = np.int0(corners)
    
    img_shi_tomasi = image.copy()
    for corner in corners:
        x, y = corner.ravel()
        cv2.circle(img_shi_tomasi, (x, y), 3, (0, 255, 0), -1)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(img_harris)
    axes[0].set_title('Harris Corners')
    axes[0].axis('off')
    
    axes[1].imshow(img_shi_tomasi)
    axes[1].set_title('Shi-Tomasi Corners')
    axes[1].axis('off')
    plt.show()
    
    return corners

### 2. SIFT and SURF Features

In [ ]:
def sift_detection(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    
    # SIFT (Scale-Invariant Feature Transform)
    sift = cv2.SIFT_create()
    keypoints, descriptors = sift.detectAndCompute(gray, None)
    
    # Draw keypoints
    img_sift = cv2.drawKeypoints(image, keypoints, None, 
                                 flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    
    print(f"Found {len(keypoints)} keypoints")
    print(f"Descriptor shape: {descriptors.shape if descriptors is not None else 'None'}")
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_sift)
    plt.title(f'SIFT Features - {len(keypoints)} Keypoints')
    plt.axis('off')
    plt.show()
    
    return keypoints, descriptors

### 3. ORB Features

In [ ]:
def orb_detection(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    
    # ORB (Oriented FAST and Rotated BRIEF)
    orb = cv2.ORB_create(nfeatures=1000)
    keypoints, descriptors = orb.detectAndCompute(gray, None)
    
    # Draw keypoints
    img_orb = cv2.drawKeypoints(image, keypoints, None, 
                                color=(0, 255, 0), 
                                flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    
    print(f"Found {len(keypoints)} ORB keypoints")
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_orb)
    plt.title(f'ORB Features - {len(keypoints)} Keypoints')
    plt.axis('off')
    plt.show()
    
    return keypoints, descriptors

# Feature Matching <a id="feature_ matching"></a>
Go to [content](#content)

### 1. Brute-Force Matching

In [ ]:
def feature_matching(img1, img2):
    # Convert to grayscale if needed
    gray1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY) if len(img1.shape) == 3 else img1
    gray2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY) if len(img2.shape) == 3 else img2
    
    # Initialize ORB detector
    orb = cv2.ORB_create()
    
    # Find keypoints and descriptors
    kp1, des1 = orb.detectAndCompute(gray1, None)
    kp2, des2 = orb.detectAndCompute(gray2, None)
    
    # Brute-Force Matcher
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)
    
    # Draw top 50 matches
    img_matches = cv2.drawMatches(img1, kp1, img2, kp2, 
                                  matches[:50], None, 
                                  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    
    plt.figure(figsize=(15, 10))
    plt.imshow(img_matches)
    plt.title(f'Feature Matching - {len(matches)} matches')
    plt.axis('off')
    plt.show()
    
    return matches, kp1, kp2

### 2. FLANN-based Matching

In [ ]:
def flann_matching(img1, img2):
    gray1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY) if len(img1.shape) == 3 else img1
    gray2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY) if len(img2.shape) == 3 else img2
    
    # SIFT detector
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)
    
    # FLANN parameters
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(des1, des2, k=2)
    
    # Lowe's ratio test
    good_matches = []
    for m, n in matches:
        if m.distance < 0.7 * n.distance:
            good_matches.append(m)
    
    # Draw matches
    img_matches = cv2.drawMatches(img1, kp1, img2, kp2, 
                                  good_matches[:50], None, 
                                  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    
    plt.figure(figsize=(15, 10))
    plt.imshow(img_matches)
    plt.title(f'FLANN Matching - {len(good_matches)} good matches')
    plt.axis('off')
    plt.show()
    
    return good_matches, kp1, kp2

# Geometric Transformations <a id="geometric_transformations"></a>
Go to [content](#content)

### 1. Affine Transformations

In [ ]:
def affine_transforms(image):
    rows, cols = image.shape[:2]
    
    # Translation
    M_trans = np.float32([[1, 0, 50], [0, 1, 50]])
    translated = cv2.warpAffine(image, M_trans, (cols, rows))
    
    # Rotation
    M_rot = cv2.getRotationMatrix2D((cols/2, rows/2), 45, 0.5)
    rotated = cv2.warpAffine(image, M_rot, (cols, rows))
    
    # Scaling
    scaled = cv2.resize(image, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_LINEAR)
    
    # Shear
    M_shear = np.float32([[1, 0.5, 0], [0.5, 1, 0]])
    sheared = cv2.warpAffine(image, M_shear, (int(cols*1.5), int(rows*1.5)))
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    images = [translated, rotated, scaled, sheared]
    titles = ['Translation', 'Rotation + Scaling', 'Scaling', 'Shear']
    
    for ax, img, title in zip(axes.flat, images, titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    plt.show()

### 2. Perspective Transformations

In [ ]:
def perspective_transform(image):
    rows, cols = image.shape[:2]
    
    # Source points (original corners)
    pts_src = np.float32([[0, 0], [cols-1, 0], [0, rows-1], [cols-1, rows-1]])
    
    # Destination points (warped corners)
    pts_dst = np.float32([[cols*0.1, rows*0.1], 
                         [cols*0.9, rows*0.05], 
                         [cols*0.05, rows*0.95], 
                         [cols*0.85, rows*0.9]])
    
    # Get perspective transform matrix
    M = cv2.getPerspectiveTransform(pts_src, pts_dst)
    transformed = cv2.warpPerspective(image, M, (cols, rows))
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(image)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    axes[1].imshow(transformed)
    axes[1].set_title('Perspective Transformed')
    axes[1].axis('off')
    plt.show()
    
    return transformed

# Image Segmentation <a id="image_segmentation"></a>
Go to [content](#content)

### 1. Region-based Segmentation

In [ ]:
def region_segmentation(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    
    # Watershed segmentation
    # Threshold and find contours
    ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Noise removal
    kernel = np.ones((3,3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # Sure background area
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    
    # Finding sure foreground area
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    ret, sure_fg = cv2.threshold(dist_transform, 0.7*dist_transform.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    
    # Finding unknown region
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    # Marker labelling
    ret, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    
    # Apply watershed
    markers = cv2.watershed(image, markers)
    image[markers == -1] = [255, 0, 0]  # Mark boundaries in red
    
    plt.figure(figsize=(10, 8))
    plt.imshow(image)
    plt.title('Watershed Segmentation')
    plt.axis('off')
    plt.show()
    
    return markers

### 2. K-means Segmentation

In [ ]:
def kmeans_segmentation(image, k=3):
    # Reshape image to 2D array
    pixel_values = image.reshape((-1, 3))
    pixel_values = np.float32(pixel_values)
    
    # Define criteria
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    
    # Apply k-means
    _, labels, centers = cv2.kmeans(pixel_values, k, None, criteria, 10, 
                                    cv2.KMEANS_RANDOM_CENTERS)
    
    # Convert back to uint8
    centers = np.uint8(centers)
    segmented_data = centers[labels.flatten()]
    segmented_image = segmented_data.reshape(image.shape)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(image)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    axes[1].imshow(segmented_image)
    axes[1].set_title(f'K-means Segmentation (k={k})')
    axes[1].axis('off')
    plt.show()
    
    return segmented_image

# Object Detection <a id="object_detection"></a>
Go to [content](#content)

### 1. Template Matching

In [ ]:
def template_matching(image, template):
    # Convert to grayscale
    img_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if len(image.shape) == 3 else image
    templ_gray = cv2.cvtColor(template, cv2.COLOR_RGB2GRAY) if len(template.shape) == 3 else template
    
    # Get template size
    h, w = templ_gray.shape
    
    # Apply template matching
    methods = ['cv2.TM_CCOEFF', 'cv2.TM_CCOEFF_NORMED', 'cv2.TM_CCORR',
               'cv2.TM_CCORR_NORMED', 'cv2.TM_SQDIFF', 'cv2.TM_SQDIFF_NORMED']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for idx, method_name in enumerate(methods):
        method = eval(method_name)
        result = cv2.matchTemplate(img_gray, templ_gray, method)
        
        # Find best match
        min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)
        
        # For SQDIFF methods, min is best; for others, max is best
        if method in [cv2.TM_SQDIFF, cv2.TM_SQDIFF_NORMED]:
            top_left = min_loc
        else:
            top_left = max_loc
            
        bottom_right = (top_left[0] + w, top_left[1] + h)
        
        # Draw rectangle
        img_copy = image.copy()
        cv2.rectangle(img_copy, top_left, bottom_right, (0, 255, 0), 2)
        
        axes[idx].imshow(img_copy)
        axes[idx].set_title(f'{method_name}')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

### 2. HOG + SVM Object Detection

In [ ]:
def hog_detection(image):
    # Create HOG descriptor
    hog = cv2.HOGDescriptor()
    hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
    
    # Detect people
    boxes, weights = hog.detectMultiScale(image, winStride=(4, 4),
                                          padding=(8, 8), scale=1.05)
    
    # Draw detections
    img_copy = image.copy()
    for (x, y, w, h) in boxes:
        cv2.rectangle(img_copy, (x, y), (x+w, y+h), (0, 255, 0), 2)
    
    plt.figure(figsize=(10, 8))
    plt.imshow(img_copy)
    plt.title(f'Detected {len(boxes)} objects')
    plt.axis('off')
    plt.show()
    
    return boxes

# Motion Analysis <a id="motion_analysis"></a>
Go to [content](#content)

### 1. Optical Flow

In [ ]:
def optical_flow(video_path):
    cap = cv2.VideoCapture(video_path)
    
    # Read first frame
    ret, frame1 = cap.read()
    prvs = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    hsv = np.zeros_like(frame1)
    hsv[..., 1] = 255
    
    while True:
        ret, frame2 = cap.read()
        if not ret:
            break
            
        next_frame = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
        
        # Calculate optical flow using Farneback method
        flow = cv2.calcOpticalFlowFarneback(prvs, next_frame, None,
                                           0.5, 3, 15, 3, 5, 1.2, 0)
        
        # Convert flow to HSV
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        hsv[..., 0] = ang * 180 / np.pi / 2
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
        
        # Convert to RGB
        rgb_flow = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
        
        # Display
        cv2.imshow('Optical Flow', rgb_flow)
        
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break
            
        prvs = next_frame
    
    cap.release()
    cv2.destroyAllWindows()

### 2. Background Subtraction

In [ ]:
def background_subtraction(video_path):
    # Create background subtractor
    # MOG2 or KNN
    backSub = cv2.createBackgroundSubtractorMOG2()
    # backSub = cv2.createBackgroundSubtractorKNN()
    
    cap = cv2.VideoCapture(video_path)
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        # Apply background subtraction
        fgMask = backSub.apply(frame)
        
        # Display
        cv2.imshow('Original', frame)
        cv2.imshow('Foreground Mask', fgMask)
        
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()